In [11]:
import re
from pathlib import Path
import shutil

In [19]:
source = Path("/Users/morganr/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/Flaubert_brute/Bouvard et Pécuchet.txt") #chemin vers le fichier brut à nettoyer
destination = source.with_name(source.stem + "_clean.txt")

In [20]:
def est_ligne_parasite(ligne):
    ligne = ligne.strip()

    if not ligne:
        return False

    if re.match(
        r'^(table des matières du titre|liste des titres|table)$',
        ligne,
        flags=re.IGNORECASE
    ):
        return True

    if re.match(r'^chapitre\s+[IVXLCDM]+\s*$', ligne, flags=re.IGNORECASE):
        return True

    if re.match(r'^(livre|partie)\s+[IVXLCDM]+\s*$', ligne, flags=re.IGNORECASE):
        return True

    if re.match(r"^[A-ZÀÂÄÇÉÈÊËÎÏÔÖÙÛÜŸÆŒ0-9' -]{5,}$", ligne):
        mots = ligne.split()
        if len(mots) >= 2:
            return True

    return False


def proteger_ponctuation_dans_guillemets(texte):
    def repl(match):
        contenu = match.group(0)
        contenu = contenu.replace(".", "<QUOTE_DOT>")
        contenu = contenu.replace("!", "<QUOTE_EXCL>")
        contenu = contenu.replace("?", "<QUOTE_QMARK>")
        return contenu

    return re.sub(r'"[^"]*"', repl, texte)


def restaurer_ponctuation_dans_guillemets(texte):
    texte = texte.replace("<QUOTE_DOT>", ".")
    texte = texte.replace("<QUOTE_EXCL>", "!")
    texte = texte.replace("<QUOTE_QMARK>", "?")
    return texte


def nettoyer_texte(texte):
    # 1. Normalisation typographique
    texte = texte.replace("’", "'")
    texte = texte.replace("“", '"').replace("”", '"')
    texte = texte.replace("«", '"').replace("»", '"')
    texte = texte.replace("–", "-").replace("—", "-")

    # 2. Suppression des lignes parasites
    lignes = texte.splitlines()
    lignes_filtrees = [ligne for ligne in lignes if not est_ligne_parasite(ligne)]
    texte = "\n".join(lignes_filtrees)

    # 3. Réparation d’artefacts simples
    texte = re.sub(r'(\w)-\s+(\w)', r'\1\2', texte)
    texte = re.sub(r'\s+([.,;:?!])', r'\1', texte)

    # 4. Compactage
    texte = re.sub(r'\s+', ' ', texte).strip()

    # 5. Protection des points de suspension
    texte = texte.replace("...", "<ELLIPSIS>")

    # 6. Protection des abréviations
    abreviations = [
        "M.", "MM.", "Mr.", "Mme.", "Mmes.", "Mlle.",
        "Dr.", "Pr.", "St.", "Ste.", "Sr.", "Jr.",
        "etc.", "cf.", "chap.", "vol.", "t."
    ]

    for abbr in abreviations:
        texte = texte.replace(abbr, abbr.replace(".", "<ABBR_DOT>"))

    # 7. Protection de la ponctuation dans les guillemets
    texte = proteger_ponctuation_dans_guillemets(texte)

    # 8. Découpage après fin de phrase hors guillemets
    texte = re.sub(r'([.!?])\s+', r'\1\n', texte)

    # 9. Découpage avant tiret de dialogue
    texte = re.sub(r'\s*-\s+(?=")', r'\n- ', texte)

    # 10. Restauration
    texte = restaurer_ponctuation_dans_guillemets(texte)
    texte = texte.replace("<ABBR_DOT>", ".")
    texte = texte.replace("<ELLIPSIS>", "...")

    # 11. Nettoyage final
    lignes = [ligne.strip() for ligne in texte.split('\n') if ligne.strip()]
    texte = '\n'.join(lignes)

    return texte


try:
    with open(source, "r", encoding="utf-8") as f:
        contenu_brut = f.read()

    contenu_propre = nettoyer_texte(contenu_brut)

    with open(destination, "w", encoding="utf-8") as f:
        f.write(contenu_propre)

    print(f"Nettoyage terminé. Le fichier propre a été enregistré ici : {destination}")

except FileNotFoundError:
    print("Erreur : le fichier source est introuvable. Vérifie le nom, le chemin et l'extension .txt.")

Nettoyage terminé. Le fichier propre a été enregistré ici : /Users/morganr/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/Flaubert_brute/Bouvard et Pécuchet_clean.txt
